# Multimodal Models

In this chapter we explore models that can process multiple modes of data (text, images, ...).

## How Multimodality works

To understand how a multimodal model can output a detailed and correct textual description of an unseen image, let's break down the process.

### 1. **Training Phase**

#### A. **Data Collection**

The first step involves collecting a large dataset of paired images and their corresponding textual descriptions. These pairs are crucial for teaching the model to understand the relationship between visual content and language.



#### B. **Image and Text Encoders**

Two separate neural networks are trained:

- **Image Encoder**: Typically a convolutional neural network (CNN) like ResNet or Vision Transformer (ViT) is used. This encoder processes each image and converts it into an **embedding** (fixed-length vector representation).

- **Text Encoder**: A transformer-based model processes the textual descriptions and converts them into vector representations.

#### C. **Contrastive Learning**

During training, the model learns to align the image and text embeddings in a **shared latent space** using a **contrastive loss function**. The goal is to bring the embeddings of matching image-text pairs closer together while pushing apart the embeddings of non-matching pairs. This allows the model to understand which textual descriptions are appropriate for specific images.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/clip-contrastrive-training.jpg" style="max-width: 600px; max-height: 500px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    CLIP contrastive pre-training — from Figure 1 of Radford et al., <a href="https://arxiv.org/abs/2103.00020">"Learning Transferable Visual Models From Natural Language Supervision"</a> (2021)
  </div>
</div>


<div style="border:2px solid gray; padding:10px; width: 95%;">

ℹ **Contrastive Learning**

Contrastive Learning is a form of supervised or self-supervised learning used in machine learning to learn effective representations of data by contrasting positive pairs against negative pairs. In the context of machine learning models dealing with images and text:

- **Positive pairs** consist of instances that are similar or related in some way, such as an image and its correct textual description.
- **Negative pairs** involve instances that are not related, such as an image and an incorrect textual description.

The main goal of contrastive learning is to ensure that the representations (or embeddings) of similar or related items are close to each other in the embedding space, while those of unrelated items are far apart. 



### 2. **Inference Phase**

When the trained model encounters a new, unseen image, here's how it can generate a detailed and correct textual description:

#### A. **Image Encoding**

- **Extract Features**: The unseen image is passed through the trained image encoder to extract its feature representation (embedding).

#### B. **Text Generation**

- **Latent Space Navigation**: The image embedding is then used as a condition to influence the text generation process. This can be done in several ways, but a common approach involves using the image embedding to guide a language model.
  
- **Language Model**: A multimodal LLM (like _GPT-4_ or _LlaVA_) takes the image embedding as an additional input. This allows the language model to generate text that is coherent with the visual features represented by the embedding.



### Why It Works

- **Shared Latent Space**: The alignment learned during training allows the model to understand the semantic connections between visual elements and language.
- **Rich Training Data**: Exposure to a diverse and extensive dataset helps the model generalize to new images by drawing on similar examples it has seen during training.
- **Powerful Language Model**: Using an advanced language model ensures that the generated text is not only relevant but also grammatically correct and contextually rich.



## Example: Qwen3.5

[**Qwen3.5**](https://ollama.com/library/qwen3.5) is a natively multimodal model family developed by Alibaba: every Qwen3.5 model accepts images as well as text, so no separate vision-language (VL) variant is needed. It excels at image understanding, visual reasoning, and multilingual capabilities.

In [ ]:
from ai_dojo import show

We are using _Ollama_ to pull and serve the model.

In [ ]:
import ollama

In [ ]:
ollama_model_name = "qwen3.5:9b"

In [ ]:
ollama.pull(ollama_model_name)

We are pulling an 8 billion parameter variant of the model.

Here are some example images - a photo and a more complex diagram. Let's see whether _Qwen3.5_ can correctly describe them and extract information from the picture.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/alpacas.jpg" style="max-width: 700px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Alpacas at Sillustani, Peru — photo by Christophe Meneboeuf, <a href="https://creativecommons.org/licenses/by-sa/2.5/">CC BY-SA 2.5</a>, via <a href="https://commons.wikimedia.org/wiki/File:Alpacas_Sillustani_(pixinn.net).jpg">Wikimedia Commons</a>
  </div>
</div>

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/camelidae.jpg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Camelidae phylogenetic tree — Figure 1 from Hoter et al., <a href="https://doi.org/10.3389/fgene.2019.00588">"Cellular and Molecular Adaptation of Arabian Camel to Heat Stress"</a>, Frontiers in Genetics (2019), <a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>
  </div>
</div>

In order to pass the images to the model via the Ollama API, we need to encode them as strings in base64 encoding.

In [ ]:
import base64
from PIL import Image
import io

def image_to_base64(image_path):
    # Open the image file
    with Image.open(image_path) as img:
        # Convert the image to RGB format (optional, depending on use case)
        img = img.convert('RGB')
        
        # Save the image to a bytes buffer
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG')  # You can change JPEG to PNG or other formats depending on your image
        
        # Get the byte data from the buffer
        byte_data = buffer.getvalue()
        
        # Encode the byte data to a Base64 string
        base64_str = base64.b64encode(byte_data).decode('utf-8')
        
        return base64_str


In [ ]:
encoded_image = image_to_base64("graphics/alpacas.jpg")
encoded_image[:100]

We are now ready to prompt the model.

In [ ]:
response = ollama.generate(
    model=ollama_model_name,
    prompt="What is in this picture?",
    images=[encoded_image]
)


In [ ]:
show.image(
    "graphics/alpacas.jpg",
    caption=f'"{response["response"]}"',
)

In [ ]:
encoded_image = image_to_base64("graphics/camelidae.jpg")


In [ ]:
response = ollama.generate(
    model=ollama_model_name,
    prompt="According to the image, which species are found in South America? List the species names.",
    images=[encoded_image]
)

show.image(
    "graphics/camelidae.jpg",
    caption=f'"{response["response"]}"',
)

In [ ]:
response = ollama.generate(
    model=ollama_model_name,
    prompt="Which two subfamilies are shown in the phylogenetic tree?",
    images=[encoded_image]
)

show.image(
    "graphics/camelidae.jpg",
    caption=f'"{response["response"]}"',
)

<div style="border:2px solid gray; padding:10px; width: 95%;">

<b>🫳 Hands-on: Try your own image</b>

Now it's your turn! Try the multimodal model on your own images.

**Your task:**
1. Load an image from a URL or upload a local image file
2. Ask the model a question about the image
3. Experiment with different types of images (photos, diagrams, charts, screenshots, etc.)

</div>

**Option 1: Load an image from a URL**

In [ ]:
import requests
from io import BytesIO

def image_from_url(url: str) -> str:
    """Fetch an image from a URL and return it as a base64 string."""
    response = requests.get(url)
    response.raise_for_status()
    image_bytes = response.content
    return base64.b64encode(image_bytes).decode("utf-8")

# Example: load a local image (photo: tabby cat by pasja1000, CC0, via Wikimedia Commons)
with open("graphics/cat.jpg", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

**Option 2: Upload an image using a widget**

Run the cell below to display an upload button, then click it to select an image from your computer.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(accept="image/*", multiple=False)
display(upload)

In [ ]:
# After uploading, run this cell to get the base64 string
if upload.value:
    uploaded_file = upload.value[0]
    image_b64 = base64.b64encode(uploaded_file["content"]).decode("utf-8")
    print(f"Image uploaded: {uploaded_file['name']}")
else:
    print("No image uploaded yet. Please upload an image first.")

**Query the model with your image**

Write your own prompt and call `ollama.chat()` to ask the model about your image. Use the examples above as a reference.

In [ ]:
# TODO: Write your prompt
your_prompt = ...

# TODO: Call ollama.chat() with your prompt and image_b64
response = ...

# TODO: Display the response
...

## Sources

- [📄 **Visual Instruction Tuning**](https://arxiv.org/pdf/2304.08485)

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_